# 费用报销分析

本笔记本演示如何创建使用插件处理本地收据图像中的旅行费用、生成费用报销邮件并使用饼图可视化费用数据的智能体。智能体根据任务上下文动态选择函数。

步骤:
1. OCR 智能体处理本地收据图像并提取旅行费用数据。
2. 邮件智能体生成费用报销邮件。

### 旅行费用场景示例:
想象您是一名前往另一个城市参加商务会议的员工。您的公司有一项政策,可以报销所有合理的旅行相关费用。以下是潜在旅行费用的细分:
- 交通:
从您所在城市到目的地城市的往返机票。
往返机场的出租车或网约车服务。
目的地城市的本地交通(如公共交通、租车或出租车)。

- 住宿:
在靠近会议场地的中档商务酒店住宿三晚。

- 餐饮:
根据公司每日津贴政策,每日的早餐、午餐和晚餐津贴。

- 杂项费用:
机场停车费。
酒店互联网接入费用。
小费或小额服务费。

- 文档:
您提交所有收据(航班、出租车、酒店、餐饮等)和填写的费用报告以进行报销。

## 导入所需的库

导入笔记本所需的库和模块。

In [1]:
import os
from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential
from semantic_kernel.kernel import Kernel
from semantic_kernel.agents import AgentGroupChat
from openai import AsyncOpenAI
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat


from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.agents.strategies import SequentialSelectionStrategy, DefaultTerminationStrategy
from semantic_kernel.contents.chat_message_content import ChatMessageContent
from semantic_kernel.contents import ImageContent, TextContent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings

from semantic_kernel.functions import kernel_function, KernelArguments
from pydantic import BaseModel, Field
from typing import List
from azure.ai.inference.models import SystemMessage, UserMessage, TextContentItem, ImageContentItem, ImageUrl, ImageDetailLevel

load_dotenv()

True

In [2]:
def _create_kernel_with_chat_completion(service_id: str) -> Kernel:
    kernel = Kernel()
   
    client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.inference.ai.azure.com/")
    kernel.add_service(
        OpenAIChatCompletion(
            ai_model_id="gpt-4o-mini",
            async_client=client,
            service_id="open_ai"
        )
    )

    kernel.add_service(
        OpenAIChatCompletion(
            ai_model_id="gpt-4o",
            async_client=client,
            service_id="gpt-4o"
        )
    )

    return kernel

 ## 定义费用模型

 为单个费用创建 Pydantic 模型,并创建 ExpenseFormatter 类将用户查询转换为结构化的费用数据。

 每个费用将以以下格式表示:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [3]:
class Expense(BaseModel):
    date: str = Field(..., description="费用日期,格式为 dd-MMM-yyyy")
    description: str = Field(..., description="费用描述")
    amount: float = Field(..., description="费用金额")
    category: str = Field(..., description="费用类别(例如:交通、餐饮、住宿、杂项)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="包含费用详细信息的原始查询输入")
    
    def parse_expenses(self) -> List[Expense]:
        """
        将原始查询解析为 Expense 对象列表。
        预期格式:用分号分隔的 "date|description|amount|category"。
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] 解析错误: '{expense_str}' 中的数据无效: {e}")
        return expense_list

## 定义智能体 - 生成邮件

创建智能体类以生成用于提交费用报销的邮件。
- 此智能体使用 `kernel_function` 装饰器定义一个函数,用于生成提交费用报销的邮件。
- 它计算费用的总金额,并将详细信息格式化为邮件正文。

In [4]:
class ExpenseEmailAgent:

    @kernel_function(description="生成向财务团队提交费用报销的邮件")
    async def generate_expense_email(expenses):
        total_amount = sum(expense['amount'] for expense in expenses)
        email_body = "尊敬的财务团队,\n\n"
        email_body += "请查看以下我的费用报销详细信息:\n\n"
        for expense in expenses:
            email_body += f"- {expense['description']}: ${expense['amount']}\n"
        email_body += f"\n总金额: ${total_amount}\n\n"
        email_body += "所有费用的收据已附上供您参考。\n\n"
        email_body += "谢谢,\n[您的姓名]"
        return email_body

# 从收据图像中提取旅行费用的智能体

创建智能体类以从收据图像中提取旅行费用。
- 此智能体使用 `kernel_function` 装饰器定义一个函数,用于从收据图像中提取旅行费用。
- 使用 OCR(光学字符识别)将收据图像转换为文本,并提取相关信息,如日期、描述、金额和类别。

In [5]:
class OCRAgentPlugin:
    def __init__(self):
        self.client = ChatCompletionsClient(
            endpoint="https://models.inference.ai.azure.com/",
            credential=AzureKeyCredential(os.environ.get("GITHUB_TOKEN")),
        )
        self.model_name = "gpt-4o"

    @kernel_function(description="使用 gpt-4o 模型从 receipt.jpg 中提取结构化的旅行费用数据")
    def extract_text(self, image_path: str = "receipt.jpg") -> str:
        try:
            image_url_str = str(ImageUrl.load(image_file=image_path, image_format="jpg", detail=ImageDetailLevel.HIGH))

            prompt = (
                "您是一位专业的 OCR 助手,专门从收据图像中提取结构化数据。 "
                "分析提供的收据图像,并以以下格式提取旅行相关的费用详细信息: "
                "'date|description|amount|category',用分号分隔。 "
                "遵循以下规则: "
                "- 日期:将日期(例如 '4/4/22')转换为 'dd-MMM-yyyy' 格式(例如 '04-Apr-2022')。 "
                "- 描述:提取项目名称(例如 'Carlson's Drylawn', 'Peigs transaction Probiotics')。 "
                "- 金额:使用数值(例如从 '$4.50' 或 '4.50 dollars' 中提取 '4.50')。 "
                "- 类别:根据上下文推断(例如食品为 'Meals',旅行为 'Transportation',住宿为 'Accommodation',其他为 'Miscellaneous')。 "
                "忽略总计、小计或服务费,除非它们是逐项列出的费用。 "
                "如果未发现费用,返回 'No expenses detected'。 "
                "仅返回结构化数据,不要返回其他文本。"
            )
            response = self.client.complete(
                messages=[
                    SystemMessage(content=prompt),
                    UserMessage(content=[
                        TextContentItem(text="从此收据图像中提取旅行费用。"),
                        ImageContentItem(image_url=ImageUrl(url=image_url_str))
                    ])
                ],
                model=self.model_name,
                temperature=0.1,
                max_tokens=2048
            )
            extracted_text = response.choices[0].message.content
            return extracted_text
        except Exception as e:
            error_msg = f"[LOG] OCR 插件:处理图像时出错: {str(e)}"
            print(error_msg)
            return error_msg

## 处理费用

定义异步函数以处理费用,通过创建和注册必要的智能体,然后调用它们。
- 此函数通过加载环境变量、创建必要的智能体并将它们注册为插件来处理费用。
- 它创建一个包含两个智能体的群组聊天,并发送提示消息以根据费用数据生成邮件和饼图。
- 它处理聊天调用期间发生的任何错误,并确保智能体的正确清理。

In [6]:
async def process_expenses():
    load_dotenv()
    settings_slm = OpenAIChatPromptExecutionSettings(service_id="gpt-4o")
    settings_llm = OpenAIChatPromptExecutionSettings(service_id="open_ai")  # 修复 service_id 中的拼写错误
    
    ocr_agent = ChatCompletionAgent(
        kernel=_create_kernel_with_chat_completion("ocrAgent"),
        name="ocr_agent",
        instructions="使用 'ocrAgent' 插件中的 'extract_text' 函数从提示中的收据图像中提取旅行费用数据。以 'date|description|amount|category' 格式返回数据,用分号分隔。",
        arguments=KernelArguments(settings=settings_slm)
    )
    
       
    email_agent = ChatCompletionAgent(
            kernel=_create_kernel_with_chat_completion("expenseEmailAgent"),
            name="email_agent",
            instructions="从上一个智能体获取旅行费用数据,并使用 'expenseEmailAgent' 插件中的 'generate_expense_email' 函数生成专业的费用报销邮件,然后转发数据。",
            arguments=KernelArguments(
                settings=settings_llm)
        )


    kernel = Kernel()

    # 使用同一文件夹中 receipt.jpg 的固定路径
    image_path = "./receipt.jpg"
    
    # 创建包含文本和图像内容的结构化消息,用于 OCR 处理
    image_url_str = f"file://{image_path}"
    
    # 使用正确的多模态内容格式
    user_message = ChatMessageContent(
        role=AuthorRole.USER,
        items=[
            TextContent(text="""
            请从此收据图像中提取原始文本,重点关注旅行费用,如日期、描述、金额和类别(例如:交通、住宿、餐饮、杂项)。
            然后生成专业的费用报销邮件。
                        """),
            ImageContent.from_image_file(path=image_path)
        ]
    )

    # 向内核注册插件
    kernel.add_plugin(OCRAgentPlugin(), plugin_name="ocrAgent")
    kernel.add_plugin(ExpenseEmailAgent(), plugin_name="expenseEmailAgent")

    # 创建群组聊天
    chat = AgentGroupChat(
        agents=[ocr_agent, email_agent],
        selection_strategy=SequentialSelectionStrategy(initial_agent=ocr_agent),
        termination_strategy=DefaultTerminationStrategy(maximum_iterations=1)
    )

    # 添加带有提示的用户消息
    await chat.add_chat_message(user_message)
    print(f"# 用户消息已添加到聊天中,包含收据图像")

    async for content in chat.invoke():
        print(f"# 智能体 - {content.name or '*'}: '{content.content}'")


## 主函数

定义主函数以清除控制台并异步运行 `process_expenses` 函数。

In [9]:
async def main():
    # 清除控制台
    os.system('cls' if os.name=='nt' else 'clear')

    # 运行异步智能体代码
    await process_expenses()

await main()

# User message added to chat with receipt image
# Agent - ocr_agent: 'The receipt primarily seems to capture costs for meals and beverages. Below is the extracted travel expense data:

**Travel Expense Data:**  
`2 May '22|Meals at restaurant|75.15|Meals`

---

**Professional Expense Claim Email Draft:**  

**Subject:** Expense Claim for Meals – 2 May 2022  

Dear [Recipient's Name],  

I am submitting an expense claim for a meal incurred during a business-related trip. Below are the details:  

- **Date:** 2 May 2022  
- **Expense Description:** Meals at a restaurant  
- **Amount:** $75.15  
- **Category:** Meals  

Please find the attached receipt for your reference. Kindly process the reimbursement at your earliest convenience. Let me know if you require additional information.  

Thank you for your assistance.  

Best regards,  
[Your Name]  
[Your Contact Information]  

Let me know if you need further revisions or additional details!'
